In [ ]:
using LinearAlgebra: tr, I
using Parameters
using StaticArrays, HCubature, QuadGK
import DifferentialEquations
using Interpolations
using NLsolve
using ForwardDiff: gradient

solve(f, x0) = nlsolve(f, x0; inplace=false, show_trace=true, method=:newton, ftol=1e-6)
saddle_points(f, x0) = solve(x -> gradient(f, x), x0)
∫dk(f; d=2,atol=1e-5,rtol=1e-3) = hcubature(f, repeat([-π], d), repeat([π], d); atol=atol, rtol=rtol)[1] / (2π)^d
∫dt(t0,tf,f;atol=1e-5,rtol=1e-3) = quadgk(f, t0, tf; atol=atol, rtol=rtol)[1]

In [ ]:
"""
Cubic Anderson model parameters (units of tc)
"""
@with_kw mutable struct PAM
    V₀::Float64 = 0.    # Hybridization strength
    tf::Float64 = 0.    # F-electron hopping
    ε₀f::Float64 = 0.   # F-electron ground state e
    μ::Float64 = 0.     # Chemical potential
    β::Float64 = Inf    # Inverse temperature
  
    b::Float64 = NaN       # Saddle-point condensed slave boson
    λ::Float64 = NaN       # Saddle-point lagrange multiplier
end

"""
Slave-boson mean-field free energy at T -> 0
"""
function FreeEnergy(x::Vector, p::PAM)
  b, λ = x

  function bands(k::SVector)
    c = -2.0 * sum(cos, k) - p.μ
    f = -2.0p.tf * sum(cos, k) * abs2(b) + λ - p.μ
    h = (p.V₀ * b)

    σ = 0.5 * (c + f)
    Δ = sqrt((0.5 * (c - f))^2 + abs2(h))

    return σ - Δ, σ - Δ, σ + Δ, σ + Δ
  end

  return (λ - p.ε₀f) * (abs2(b) - 1) + ∫dk(k -> sum(bands(k)[1:2]))
end

"""
Slave-boson mean-field occupation of the fermionic fields
"""
function G0L(k::SVector, p::PAM, b::Number=p.b, λ::Number=p.λ)::Matrix{ComplexF64}
  c = -2.0 * sum(cos, k) - p.μ
  f = -2.0p.tf * sum(cos, k) * abs2(b) + λ - p.μ
  h = (p.V₀ * b)

  σ = 0.5 * (c + f)
  δ = 0.5 * (c - f)
  Δ = sqrt(δ^2 + abs2(h))
  return [[(1 - δ/Δ) -h/Δ] ; [-h'/Δ (1 + δ/Δ)]]
end

"""
Slave-boson mean-field free energy at T -> 0
"""
function sbmf_equations(x::Vector, p::PAM)
  b, λ = x
  G0 = ∫dk(k-> G0L(k, p, b, λ))
  return [(λ - p.ε₀f) * b + p.V₀ * G0[2,1], abs2(b) + G0[2,2] - 1]
end

In [ ]:
# Parameters
pam_0 = PAM(V₀=1.0, tf=-0.0, ε₀f=-1.5)

# Find the fields that minimize the Free Energy
@show solution = saddle_points(x -> FreeEnergy(x, pam_0), [0.2,1e-1])
pam_0.b, pam_0.λ = solution.zero

In [ ]:
# Parameters
pam_0 = PAM(V₀=1.0, tf=-0.0, ε₀f=-1.5)

# Solve the slave-boson mean-field equations
@show solution = solve(x -> sbmf_equations(x, pam_0), [0.2,1e-1])
pam_0.b, pam_0.λ = solution.zero

In [ ]:
using Test

@testset "Initial conditions" begin
  G0L_ = ∫dk(k -> G0L(k, pam_0, pam_0.b, pam_0.λ))
  
  # λ * b + V cf = 0
  @test G0L_[1,2] * pam_0.V₀ + (pam_0.λ - pam_0.ε₀f) * pam_0.b ≈ 0.0 atol=1e-3
  
  # f^2 + b^2 = 1
  @test G0L_[2,2] + pam_0.b^2 ≈ 1.0 atol=1e-3
  
  # Half filling
  @test G0L_[1,1] + G0L_[2,2] ≈ 2
end

In [ ]:
# Photon occupation
function ξ(t; Ω=1,tc=5)
  return (Ω^2 / (2π))^(1/4) * exp(-(t - tc)^2 * (Ω^2 / 4))
end

const Γ = sqrt(0.3)

"""
Time-dependent slave-boson mean-field equations with photon coupling
"""
function f′(u, h, params::Tuple{PAM, Interpolations.GriddedInterpolation}, t::Float64)
  println("At t: $t")
  
  p = params[1]
  λ = params[2]
  b(t′) = h(params, t′)
  
  # G<(t,t)
  function GL(k::SVector)::Matrix{ComplexF64}
    @assert t >= 0 "time ordering not yet implemented"
    U = exp(-1.0im * ∫dt(0, t, t′ -> [
          [-2.0 * sum(cos, k) - p.μ (p.V₀ * b(t′))']; 
          [(p.V₀ * b(t′)) -2.0p.tf * sum(cos, k) * abs2(b(t′)) + λ(t) - p.μ]]::Matrix{ComplexF64}))

    return U * G0L(k, p, b(0.), λ(0.)) * U'
  end

  # solve for λ: b^2 + <f^† f> = 1
  λ.coefs[end] = nlsolve(λ′ -> begin
      λ.knots[1][end] = t
      λ.coefs[end] = λ′[1]
    
      n′ = ∫dk(GL)::Matrix{ComplexF64}
      
      abs2(b(t)) + n′[2,2] - 1 + 
        2Γ^2 * ξ(t) * (n′[1,2] * b(t)' + n′[2,1] * b(t)) * ∫dk(k -> begin
          g = GL(k)::Matrix{ComplexF64}
          g[2,2] * (g[1,2] * b(t)' + g[2,1] * b(t))
        end)::ComplexF64
      end, [λ.coefs[end-1]]; inplace=false, show_trace=true, method=:anderson, ftol=1e-4, iterations=10).zero[1]

  # solve for b: (i ∂t b + λ b + <c^† f> = 0)
  n = ∫dk(GL)::Matrix{ComplexF64}
  
  return -1.0im * ((λ(t) - p.ε₀f) * b(t) + p.V₀ * n[2,1] + 
    2Γ^2 * ξ(t) * (n[1,2]  * b(t)' + n[2,1]  * b(t)) * ∫dk(k -> begin
      g = GL(k)::Matrix{ComplexF64}
      g[1,2] - (g[1,1]  * g[2,2]  * b(t)' + g[2,1]  * g[2,1]  * b(t))
    end)::ComplexF64)
end

In [ ]:
# Because λ is found through an implicit equation (and not as the result of the time integration)
# we need to make some tricks to properly save and access this function at all `t`s

# In order to interpolate some function we need at least two points, hence we add an artificial past point
λ = interpolate(([-1.0, 0.0],), complex.([pam_0.λ, pam_0.λ]), Gridded(Linear()))

# Since the ODE solver is adaptive we should only store the points accepted by the integrator
# This can be done with a `SavingCallback` which will add a new node to the interpolator's data
cb = DifferentialEquations.SavingCallback((u,t,integrator)-> begin
    push!(λ.knots[1], t)
    push!(λ.coefs, NaN)
    nothing
    end, DifferentialEquations.SavedValues(Float64, Nothing))

# Time span
tspan = (0.0, 10.0)

# Initial condition
u0 = complex(pam_0.b)

# History functio (b(t) for t <0)
h(p,t) = complex(pam_0.b)

# Time-dependent PAM
pam_t = deepcopy(pam_0)

prob = DifferentialEquations.DDEProblem(f′, u0, h, tspan, p=(pam_t, λ))
sol = DifferentialEquations.solve(
  prob, DifferentialEquations.MethodOfSteps(DifferentialEquations.VCABM()); callback=cb)

In [ ]:
using Plots

plot(sol.t, real(sol.u) .^2; label="b^2", legend=:topleft, xlabel="t", fmt = :png)
plot!(twinx(), sol.t, ξ.(sol.t), c=:orange, label="Photon pulse amplitude", legend=:topright)